# 03 — Custom Iterators

Now we move from **understanding the iterator protocol** to actually **building our own iterators**.

A custom iterator is a Python object that we create ourselves and that follows Python's iterator protocol.

> **Progression:** `01_Iterables_and_Iterators` → `02_Iterator_Protocol` → **`03_Custom_Iterators`** → `04_Generators`

## 1. Introduction to Custom Iterators

Recall the iterator protocol:

```text
Custom Iterator
│
├── __iter__()
├── __next__()
└── StopIteration
```

A custom iterator lets us decide:
- what values it produces
- in what order
- when iteration starts
- when iteration ends
- how it maintains its state

## 2. Why Create Custom Iterators?

Python already provides iterators for lists, tuples, ranges, and many other objects. Custom iterators are useful when we want to:

- generate values according to custom rules
- control how data is traversed
- represent sequences that are not stored completely in memory
- build reusable iteration behavior
- understand how Python's iteration system works

For example, an iterator can produce `1, 3, 5, 7, 9` one value at a time instead of storing them all in a list.

## 3. Structure of a Custom Iterator

A typical custom iterator is implemented as a class:

```python
class MyIterator:

    def __iter__(self):
        return self

    def __next__(self):
        # return next value
        # raise StopIteration when finished
        pass
```

`__iter__()` returns the iterator. `__next__()` produces the next value and raises `StopIteration` when there are no more values.

In [ ]:
class MyIterator:

    def __iter__(self):
        return self

    def __next__(self):
        # return next value
        # raise StopIteration when finished
        pass

The class above shows the structure, but `__next__()` does not produce anything yet. We need to add state and iteration logic.

## 4. Building a Simple Counter Iterator

Let's build an iterator that counts from `1` up to a limit.

In [ ]:
class CountUp:
    def __init__(self, limit):
        self.current = 1
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 1
        return value

In [ ]:
counter = CountUp(5)

print(next(counter))
print(next(counter))
print(next(counter))
print(next(counter))
print(next(counter))

Expected output:

```text
1
2
3
4
5
```

Calling `next(counter)` again raises `StopIteration` because the iterator is exhausted.

In [ ]:
# The iterator is now exhausted.
# Running this line raises StopIteration.
next(counter)

## 5. Understanding Iterator State

One of the most important ideas in custom iterators is **state**.

The attribute `self.current` stores the iterator's current position.

Conceptually:

```text
CountUp
│
├── current = 1
├── next() → returns 1
├── current = 2
├── next() → returns 2
├── current = 3
└── ...
```

An iterator must maintain enough state to know what value should be returned next.

In [ ]:
counter = CountUp(5)

print("Initial state:", counter.current)

print("Value:", next(counter))
print("State:", counter.current)

print("Value:", next(counter))
print("State:", counter.current)

print("Value:", next(counter))
print("State:", counter.current)

Each call to `next()` changes the same iterator object's state. Repeated calls therefore continue from where the previous call stopped.

## 6. Using `__iter__()`

For a typical custom iterator:

```python
def __iter__(self):
    return self
```

This means the iterator object is also its own iterator.

In [ ]:
counter = CountUp(5)

print(iter(counter) is counter)

Expected output:

```text
True
```

Because `__iter__()` returns `self`, `iter(counter)` gives back the same object.

In [ ]:
counter = CountUp(5)

for number in counter:
    print(number)

Expected output:

```text
1
2
3
4
5
```

The `for` loop obtains the iterator and repeatedly calls `next()` until `StopIteration` signals that the iterator is finished.

## 7. Using `__next__()`

Each time we write:

```python
next(counter)
```

Python uses the iterator's `__next__()` method.

Inside `__next__()` we generally:

1. Check whether iteration is finished.
2. Save the current value.
3. Update the state.
4. Return the saved value.

A useful pattern is:

```python
def __next__(self):
    if finished:
        raise StopIteration

    value = current_value
    update_state()
    return value
```

In [ ]:
counter = CountUp(3)

print(next(counter))
print(next(counter))
print(next(counter))

The fourth call would find that the iterator has finished and raise `StopIteration`.

## 8. Handling `StopIteration`

A finite iterator must eventually signal that there are no more values:

```python
raise StopIteration
```

When using a `for` loop, we normally do not catch `StopIteration` ourselves; the loop handles the end automatically.

In [ ]:
for number in CountUp(3):
    print(number)

Expected output:

```text
1
2
3
```

## 9. Custom Range-Like Iterator

We already know Python's `range()` from Basic Python. Let's build a simple range-like iterator that starts at `0` and stops before the limit.

In [ ]:
class NumberRange:
    def __init__(self, stop):
        self.current = 0
        self.stop = stop

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration

        value = self.current
        self.current += 1
        return value

In [ ]:
for number in NumberRange(5):
    print(number)

Expected output:

```text
0
1
2
3
4
```

This mirrors the basic behavior of `range(5)`.

## 10. Custom Iterator with a Start and Stop

In [ ]:
class NumberRange:
    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration

        value = self.current
        self.current += 1
        return value

In [ ]:
for number in NumberRange(3, 8):
    print(number)

Expected output:

```text
3
4
5
6
7
```

Like `range(3, 8)`, the `stop` value itself is not included.

## 11. Custom Iterator with a Step

Now make the iterator more flexible by adding `step`.

In [ ]:
class NumberRange:
    def __init__(self, start, stop, step=1):
        self.current = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration

        value = self.current
        self.current += self.step
        return value

In [ ]:
for number in NumberRange(0, 10, 2):
    print(number)

Expected output:

```text
0
2
4
6
8
```

**Important limitation:** this implementation handles a positive step. We intentionally keep the first version simple rather than adding negative-step logic here.

## 12. Iterating in Reverse

We can also create an iterator that counts downward.

In [ ]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration

        value = self.current
        self.current -= 1
        return value

In [ ]:
for number in Countdown(5):
    print(number)

Expected output:

```text
5
4
3
2
1
```

This reinforces an important idea: **we control the iteration logic**.

## 13. Practical Custom Iterator Examples

Let's build a few small iterators using the same protocol.

### Even numbers

In [ ]:
class EvenNumbers:
    def __init__(self, limit):
        self.current = 0
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 2
        return value

In [ ]:
for number in EvenNumbers(10):
    print(number)

Expected output:

```text
0
2
4
6
8
10
```

### Odd numbers

In [ ]:
class OddNumbers:
    def __init__(self, limit):
        self.current = 1
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 2
        return value

In [ ]:
for number in OddNumbers(10):
    print(number)

Expected output:

```text
1
3
5
7
9
```

### Countdown by steps

In [ ]:
class Countdown:
    def __init__(self, start, step=1):
        self.current = start
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration

        value = self.current
        self.current -= self.step
        return value

In [ ]:
for number in Countdown(10, 2):
    print(number)

Expected output:

```text
10
8
6
4
2
```

## 14. Common Mistakes

### Mistake 1 — Forgetting `return self`

Incorrect:

```python
def __iter__(self):
    pass
```

Correct:

```python
def __iter__(self):
    return self
```

### Mistake 2 — Not updating state

If `self.current` never changes, every call can produce the same value:

```text
1
1
1
1
...
```

The iterator must make progress.

### Mistake 3 — Never raising `StopIteration`

If the iterator is finite, `__next__()` must eventually raise `StopIteration`.

### Mistake 4 — Updating before saving the value

Correct:

```python
value = self.current
self.current += 1
return value
```

If we increment first and then return `self.current`, the initial value is skipped.

### Mistake 5 — Reusing an exhausted iterator

A finite iterator keeps its state. Once exhausted, it stays exhausted.

In [ ]:
counter = CountUp(3)

print(list(counter))
print(list(counter))

Expected output:

```text
[1, 2, 3]
[]
```

The first `list()` consumes the iterator. To iterate again from the beginning, create a new iterator object.

In [ ]:
counter = CountUp(3)
print(list(counter))

new_counter = CountUp(3)
print(list(new_counter))

## 15. Summary

Use this mental model for a custom iterator:

```text
Custom Iterator
       │
       ▼
   __iter__()
       │
       ▼
    return self
       │
       ▼
   __next__()
       │
       ├── produce value
       ├── update state
       │
       └── when finished
               ↓
       raise StopIteration
```

### Key points

- A custom iterator is usually implemented as a class.
- It implements `__iter__()` and `__next__()`.
- `__iter__()` normally returns `self`.
- `__next__()` produces one value at a time.
- Iterator state is stored in instance attributes.
- State must change so the iterator progresses.
- `StopIteration` signals that iteration has finished.
- `for` loops automatically handle `StopIteration`.
- Once a finite iterator is exhausted, it stays exhausted.

### Curriculum progression

```text
01_Iterables_and_Iterators
        ↓
02_Iterator_Protocol
        ↓
03_Custom_Iterators       ← current
        ↓
04_Generators
```

> **Next:** `04_Generators.ipynb` will show how Python gives us a much simpler way to create iterator-like behavior without manually writing an iterator class.

Generators, `yield`, generator expressions, and iterator-vs-generator comparisons are intentionally left for the next notebooks.